In [5]:
import xgboost as xgb

print("XGBoost:", xgb.__version__)

XGBoost: 2.1.1


In [6]:
import pandas as pd
import numpy as np
import sklearn
import matplotlib
import seaborn

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)

print("\nEverything is ready!")


Pandas: 1.4.3
NumPy: 1.24.4
Scikit-learn: 1.3.2

Everything is ready!


In [7]:
summary_df = pd.read_csv("kaggle_workouts_summary.csv")

print("Shape:", summary_df.shape)
print("\nColumns:")
print(summary_df.columns.tolist())

summary_df.head()

Shape: (255, 9)

Columns:
['workout_id', 'workout_name', 'start_time', 'end_time', 'duration_mins', 'nth_workout', 'estimated_volume_kg', 'exercise_count', 'exercises_performed']


,workout_id,workout_name,start_time,end_time,duration_mins,nth_workout,estimated_volume_kg,exercise_count,exercises_performed
0,Workout_001,Leg,2024-01-01 00:00:00,2024-01-01 00:44:57,44.950000,1,1755.0,6,"Warm Up, Leg Extension (Machine), Standing Leg..."
1,Workout_002,Shoulders,2024-01-01 23:57:32,2024-01-02 00:44:35,47.050000,2,1430.0,7,"Warm Up, Standing Military Press (Barbell), Ov..."
2,Workout_003,Shoulders 💪,2024-01-03 00:02:38,2024-01-03 00:43:12,40.566667,3,1967.5,5,"Warm Up, Lat Pulldown (Cable), Shoulder Press ..."
3,Workout_004,Legs 🦵,2024-01-03 23:47:27,2024-01-04 00:42:46,55.316667,4,3590.0,7,"Warm Up, Squat (Dumbbell), Deadlift (Smith Mac..."
4,Workout_005,Back,2024-01-05 23:47:48,2024-01-06 00:48:06,60.300000,5,9049.0,6,"Warm Up, Lat Pulldown (Cable), Seated Cable Ro..."


In [8]:
import sys
print(sys.executable)


C:\Users\mmahe\anaconda3\envs\exercise-correction\python.exe


In [9]:
df = pd.read_csv("Dropout_dataset.csv") 

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (5120, 14)

Columns:
['workout_id', 'workout_start_time', 'workout_name', 'exercise_idx', 'exercise_title', 'muscle_group', 'equipment', 'set_index', 'set_type', 'weight_kg', 'reps', 'rpe', 'distance_meters', 'duration_seconds']


In [10]:
import pandas as pd
import numpy as np

data = df.copy()

print("Original shape:", data.shape)

Original shape: (5120, 14)


In [11]:
data.columns = (
    data.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print(data.columns.tolist())

['workout_id', 'workout_start_time', 'workout_name', 'exercise_idx', 'exercise_title', 'muscle_group', 'equipment', 'set_index', 'set_type', 'weight_kg', 'reps', 'rpe', 'distance_meters', 'duration_seconds']


In [12]:
print("Duplicate rows:", data.duplicated().sum())

data = data.drop_duplicates().reset_index(drop=True)

print("Shape after removing duplicates:", data.shape)


Duplicate rows: 0
Shape after removing duplicates: (5120, 14)


In [13]:
data["workout_start_time"] = pd.to_datetime(
    data["workout_start_time"],
    errors="coerce"
)

print("Invalid timestamps:", data["workout_start_time"].isna().sum())

Invalid timestamps: 0


In [14]:
data = data.dropna(
    subset=["workout_id", "workout_start_time"]
).reset_index(drop=True)

In [15]:
numeric_cols = [
    "exercise_idx",
    "set_index",
    "weight_kg",
    "reps",
    "rpe",
    "distance_meters",
    "duration_seconds"
]

for col in numeric_cols:
    data[col] = pd.to_numeric(data[col], errors="coerce")

data[numeric_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
exercise_idx,5120.0,4.263477,1.909435,1.0,3.0,4.0,6.0,11.0
set_index,5120.0,2.178711,1.054065,1.0,1.0,2.0,3.0,5.0
weight_kg,4321.0,22.428570,12.327394,0.0,15.0,20.0,29.0,120.0
reps,4652.0,17.176053,4.838275,3.0,15.0,15.0,20.0,60.0
rpe,7.0,8.714286,0.951190,7.0,8.5,9.0,9.0,10.0
distance_meters,158.0,1015.955696,865.953841,2.0,327.5,920.0,1397.5,3400.0
duration_seconds,435.0,429.308046,222.617487,20.0,323.5,408.0,600.0,3392.0


In [16]:
missing = pd.DataFrame({
    "missing_count": data.isna().sum(),
    "missing_percent": data.isna().mean() * 100
})

missing.sort_values("missing_percent", ascending=False)

,missing_count,missing_percent
rpe,5113,99.863281
distance_meters,4962,96.914062
duration_seconds,4685,91.503906
weight_kg,799,15.605469
reps,468,9.140625
workout_id,0,0.000000
workout_start_time,0,0.000000
workout_name,0,0.000000
exercise_idx,0,0.000000
exercise_title,0,0.000000


In [17]:
for col in [
    "workout_name",
    "exercise_title",
    "muscle_group",
    "equipment",
    "set_type"
]:
    print("\n==============================")
    print(col)
    print(data[col].value_counts(dropna=False).head(20))


workout_name
Chest                 1107
Back                  1041
Legs                   820
Shoulder               729
Morning workout ☀️     347
Shoulders              320
Biceps                 177
Triceps                159
Core                   153
Legs 🦵                  76
Back + biceps           29
Chest + tricep          28
Back+biceps             24
Legs 🦵🏻                 19
Circuit                 19
Chest 🧰                 17
Leg                     16
Back+lats               16
Shoulders 💪             13
Biceps 💪                10
Name: workout_name, dtype: int64

exercise_title
Warm Up                              251
Lat Pulldown (Cable)                 204
Chest Fly (Machine)                  192
Leg Extension (Machine)              171
Leg Press (Machine)                  143
Shrug (Dumbbell)                     142
Bench Press (Barbell)                139
Triceps Extension (Dumbbell)         124
Face Pull                            121
Lateral Raise (Dumbbell)    

In [18]:
workout_df = (
    data
    .groupby("workout_id")
    .agg(
        workout_start_time=("workout_start_time", "min"),
        workout_name=("workout_name", "first"),

        total_sets=("set_index", "count"),
        total_exercises=("exercise_title", "nunique"),

        total_reps=("reps", "sum"),
        avg_reps=("reps", "mean"),

        total_weight_kg=("weight_kg", "sum"),
        avg_weight_kg=("weight_kg", "mean"),

        avg_rpe=("rpe", "mean"),

        total_distance_meters=("distance_meters", "sum"),
        total_duration_seconds=("duration_seconds", "sum"),

        muscle_groups=("muscle_group", "nunique"),
        equipment_types=("equipment", "nunique"),
    )
    .reset_index()
)

workout_df.head()

,workout_id,workout_start_time,workout_name,total_sets,total_exercises,total_reps,avg_reps,total_weight_kg,avg_weight_kg,avg_rpe,total_distance_meters,total_duration_seconds,muscle_groups,equipment_types
0,Workout_001,2024-01-01 00:00:00,Leg,16,6,265.0,17.666667,102.0,8.500000,NaN,0.0,180.0,3,3
1,Workout_002,2024-01-01 23:57:32,Shoulders,19,7,315.0,17.500000,87.0,4.833333,NaN,0.0,258.0,3,3
2,Workout_003,2024-01-03 00:02:38,Shoulders 💪,13,5,205.0,17.083333,136.5,11.375000,NaN,0.0,295.0,3,3
3,Workout_004,2024-01-03 23:47:27,Legs 🦵,17,7,285.0,19.000000,192.0,12.800000,NaN,0.0,553.0,6,4
4,Workout_005,2024-01-05 23:47:48,Back,16,6,297.0,19.800000,440.0,29.333333,NaN,0.0,443.0,3,2


In [19]:
print("Workout-level shape:", workout_df.shape)

Workout-level shape: (255, 14)


In [20]:
workout_df = workout_df.sort_values(
    ["workout_start_time", "workout_id"]
).reset_index(drop=True)

In [21]:
workout_df["workout_date"] = (
    workout_df["workout_start_time"].dt.date
)

workout_df["day_of_week"] = (
    workout_df["workout_start_time"].dt.dayofweek
)

workout_df["hour"] = (
    workout_df["workout_start_time"].dt.hour
)

workout_df["month"] = (
    workout_df["workout_start_time"].dt.month
)

In [22]:
workout_df["is_weekend"] = (
    workout_df["day_of_week"] >= 5
).astype(int)

In [23]:
workout_df["duration_minutes"] = (
    workout_df["total_duration_seconds"] / 60
)

In [24]:
workout_df.head(10)

,workout_id,workout_start_time,workout_name,total_sets,total_exercises,total_reps,avg_reps,total_weight_kg,avg_weight_kg,avg_rpe,total_distance_meters,total_duration_seconds,muscle_groups,equipment_types,workout_date,day_of_week,hour,month,is_weekend,duration_minutes
0,Workout_001,2024-01-01 00:00:00,Leg,16,6,265.0,17.666667,102.0,8.500000,NaN,0.0,180.0,3,3,2024-01-01,0,0,1,0,3.000000
1,Workout_002,2024-01-01 23:57:32,Shoulders,19,7,315.0,17.500000,87.0,4.833333,NaN,0.0,258.0,3,3,2024-01-01,0,23,1,0,4.300000
2,Workout_003,2024-01-03 00:02:38,Shoulders 💪,13,5,205.0,17.083333,136.5,11.375000,NaN,0.0,295.0,3,3,2024-01-03,2,0,1,0,4.916667
3,Workout_004,2024-01-03 23:47:27,Legs 🦵,17,7,285.0,19.000000,192.0,12.800000,NaN,0.0,553.0,6,4,2024-01-03,2,23,1,0,9.216667
4,Workout_005,2024-01-05 23:47:48,Back,16,6,297.0,19.800000,440.0,29.333333,NaN,0.0,443.0,3,2,2024-01-05,4,23,1,0,7.383333
5,Workout_006,2024-01-06 23:52:58,Chest 🧰,17,7,293.0,18.312500,297.0,19.800000,NaN,0.0,277.0,2,4,2024-01-06,5,23,1,1,4.616667
6,Workout_007,2024-01-07 23:44:41,Morning workout ☀️,13,6,175.0,19.444444,135.0,15.000000,7.0,0.0,393.0,4,3,2024-01-07,6,23,1,1,6.550000
7,Workout_008,2024-01-08 23:46:51,Shoulders,22,8,420.0,20.000000,148.0,7.047619,NaN,0.0,334.0,2,4,2024-01-08,0,23,1,0,5.566667
8,Workout_009,2024-01-09 23:43:39,Chest,18,7,355.0,20.882353,301.0,17.705882,NaN,0.0,362.0,2,4,2024-01-09,1,23,1,0,6.033333
9,Workout_010,2024-01-10 23:47:41,Morning workout ☀️,1,1,0.0,NaN,0.0,NaN,NaN,0.0,3392.0,1,1,2024-01-10,2,23,1,0,56.533333


In [25]:
print(workout_df.shape)
print(workout_df.columns.tolist())

print("\nUnique workout IDs:")
print(workout_df["workout_id"].nunique())

print("\nFirst 20 workout IDs:")
print(workout_df["workout_id"].head(20).tolist())

(255, 20)
['workout_id', 'workout_start_time', 'workout_name', 'total_sets', 'total_exercises', 'total_reps', 'avg_reps', 'total_weight_kg', 'avg_weight_kg', 'avg_rpe', 'total_distance_meters', 'total_duration_seconds', 'muscle_groups', 'equipment_types', 'workout_date', 'day_of_week', 'hour', 'month', 'is_weekend', 'duration_minutes']

Unique workout IDs:
255

First 20 workout IDs:
['Workout_001', 'Workout_002', 'Workout_003', 'Workout_004', 'Workout_005', 'Workout_006', 'Workout_007', 'Workout_008', 'Workout_009', 'Workout_010', 'Workout_011', 'Workout_012', 'Workout_013', 'Workout_014', 'Workout_015', 'Workout_016', 'Workout_017', 'Workout_018', 'Workout_019', 'Workout_020']


In [26]:
print("Date range:")
print(workout_df["workout_start_time"].min())
print(workout_df["workout_start_time"].max())

Date range:
2024-01-01 00:00:00
2026-02-02 01:59:52


In [27]:
print("Original columns:")
print(df.columns.tolist())

Original columns:
['workout_id', 'workout_start_time', 'workout_name', 'exercise_idx', 'exercise_title', 'muscle_group', 'equipment', 'set_index', 'set_type', 'weight_kg', 'reps', 'rpe', 'distance_meters', 'duration_seconds']


In [28]:
workout_df = workout_df.sort_values("workout_start_time").copy()

workout_df["workout_start_time"] = pd.to_datetime(
    workout_df["workout_start_time"]
)

workout_df["days_since_previous"] = (
    workout_df["workout_start_time"].diff().dt.total_seconds() / 86400
)

workout_df[[
    "workout_id",
    "workout_start_time",
    "days_since_previous"
]].head(20)

,workout_id,workout_start_time,days_since_previous
0,Workout_001,2024-01-01 00:00:00,NaN
1,Workout_002,2024-01-01 23:57:32,0.998287
2,Workout_003,2024-01-03 00:02:38,1.003542
3,Workout_004,2024-01-03 23:47:27,0.989456
4,Workout_005,2024-01-05 23:47:48,2.000243
5,Workout_006,2024-01-06 23:52:58,1.003588
6,Workout_007,2024-01-07 23:44:41,0.994248
7,Workout_008,2024-01-08 23:46:51,1.001505
8,Workout_009,2024-01-09 23:43:39,0.997778
9,Workout_010,2024-01-10 23:47:41,1.002801


In [29]:
print("Largest gaps:")

print(
    workout_df[
        ["workout_id", "workout_start_time", "days_since_previous"]
    ]
    .sort_values("days_since_previous", ascending=False)
    .head(20)
)

Largest gaps:
      workout_id  workout_start_time  days_since_previous
32   Workout_033 2024-09-21 23:55:46           201.018866
112  Workout_113 2025-05-18 01:05:07           117.026655
19   Workout_020 2024-02-11 23:17:40            19.979410
116  Workout_117 2025-06-11 00:17:34            18.023171
130  Workout_131 2025-07-11 23:54:44            12.994583
152  Workout_153 2025-08-23 00:28:21             6.010208
101  Workout_102 2024-12-30 04:38:39             5.177558
190  Workout_191 2025-10-25 01:50:33             4.961551
102  Workout_103 2025-01-04 00:17:56             4.818947
193  Workout_194 2025-10-31 04:38:04             4.099479
242  Workout_243 2026-01-17 02:20:51             4.011956
26   Workout_027 2024-02-25 23:29:02             4.006343
162  Workout_163 2025-09-07 00:25:34             4.005069
222  Workout_223 2025-12-14 02:17:31             3.992512
235  Workout_236 2026-01-04 02:22:33             3.982477
107  Workout_108 2025-01-15 00:23:06             3.981146


In [30]:
print("Gap statistics:")
print(workout_df["days_since_previous"].describe())

Gap statistics:
count    254.000000
mean       3.004265
std       14.548302
min        0.736227
25%        0.999502
50%        1.006470
75%        2.000009
max      201.018866
Name: days_since_previous, dtype: float64


In [31]:
for days in [7, 14, 21, 30, 45, 60, 90]:
    count = (workout_df["days_since_previous"] >= days).sum()
    print(f"Gaps >= {days} days: {count}")

Gaps >= 7 days: 5
Gaps >= 14 days: 4
Gaps >= 21 days: 2
Gaps >= 30 days: 2
Gaps >= 45 days: 2
Gaps >= 60 days: 2
Gaps >= 90 days: 2


# Training Features

In [32]:
import pandas as pd
import numpy as np

# Work on a copy
model_df = workout_df.copy()

# Make sure date is datetime
model_df["workout_start_time"] = pd.to_datetime(
    model_df["workout_start_time"]
)

# Sort chronologically
model_df = model_df.sort_values("workout_start_time").reset_index(drop=True)

# Create dropout label
# 1 = next workout is 7+ days later
# 0 = next workout occurs within 7 days
model_df["next_workout_date"] = model_df["workout_start_time"].shift(-1)

model_df["days_until_next_workout"] = (
    model_df["next_workout_date"] -
    model_df["workout_start_time"]
).dt.total_seconds() / 86400

model_df["dropout"] = (
    model_df["days_until_next_workout"] >= 7
).astype(int)

# Remove final row because there is no next workout
model_df = model_df.iloc[:-1].copy()

print("Dataset shape:", model_df.shape)
print("\nDropout distribution:")
print(model_df["dropout"].value_counts())

Dataset shape: (254, 24)

Dropout distribution:
0    249
1      5
Name: dropout, dtype: int64


In [33]:
features = [
    "total_sets",
    "total_exercises",
    "total_reps",
    "avg_reps",
    "total_weight_kg",
    "avg_weight_kg",
    "avg_rpe",
    "total_distance_meters",
    "total_duration_seconds",
    "muscle_groups",
    "equipment_types",
    "duration_minutes",
    "day_of_week",
    "hour",
    "is_weekend"
]

X = model_df[features].copy()
y = model_df["dropout"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (254, 15)
y shape: (254,)


In [34]:
X = X.replace([np.inf, -np.inf], np.nan)

X = X.fillna(X.median(numeric_only=True))

print("Missing values:")
print(X.isnull().sum())

Missing values:
total_sets                0
total_exercises           0
total_reps                0
avg_reps                  0
total_weight_kg           0
avg_weight_kg             0
avg_rpe                   0
total_distance_meters     0
total_duration_seconds    0
muscle_groups             0
equipment_types           0
duration_minutes          0
day_of_week               0
hour                      0
is_weekend                0
dtype: int64


In [35]:
print("Normal:", (y == 0).sum())
print("Dropout:", (y == 1).sum())

Normal: 249
Dropout: 5


In [36]:
print(workout_df.columns.tolist())
print(workout_df.shape)

print("\nUnique workout IDs:")
print(workout_df["workout_id"].nunique())

print("\nDate range:")
print(workout_df["workout_start_time"].min())
print(workout_df["workout_start_time"].max())

['workout_id', 'workout_start_time', 'workout_name', 'total_sets', 'total_exercises', 'total_reps', 'avg_reps', 'total_weight_kg', 'avg_weight_kg', 'avg_rpe', 'total_distance_meters', 'total_duration_seconds', 'muscle_groups', 'equipment_types', 'workout_date', 'day_of_week', 'hour', 'month', 'is_weekend', 'duration_minutes', 'days_since_previous']
(255, 21)

Unique workout IDs:
255

Date range:
2024-01-01 00:00:00
2026-02-02 01:59:52


In [37]:
print(workout_df.groupby("workout_id").size().describe())

count    255.0
mean       1.0
std        0.0
min        1.0
25%        1.0
50%        1.0
75%        1.0
max        1.0
dtype: float64


In [38]:
for col in workout_df.columns:
    print(col, "->", workout_df[col].nunique())

workout_id -> 255
workout_start_time -> 255
workout_name -> 20
total_sets -> 27
total_exercises -> 10
total_reps -> 139
avg_reps -> 170
total_weight_kg -> 213
avg_weight_kg -> 227
avg_rpe -> 4
total_distance_meters -> 68
total_duration_seconds -> 211
muscle_groups -> 6
equipment_types -> 5
workout_date -> 250
day_of_week -> 7
hour -> 7
month -> 11
is_weekend -> 2
duration_minutes -> 211
days_since_previous -> 243


In [39]:
print(workout_df.columns.tolist())

['workout_id', 'workout_start_time', 'workout_name', 'total_sets', 'total_exercises', 'total_reps', 'avg_reps', 'total_weight_kg', 'avg_weight_kg', 'avg_rpe', 'total_distance_meters', 'total_duration_seconds', 'muscle_groups', 'equipment_types', 'workout_date', 'day_of_week', 'hour', 'month', 'is_weekend', 'duration_minutes', 'days_since_previous']


In [40]:
print([col for col in workout_df.columns if "drop" in col.lower()])

[]


In [41]:
print(
    workout_df[
        ["workout_id", "days_since_previous"]
    ].sort_values(
        "days_since_previous",
        ascending=False
    ).head(10)
)

      workout_id  days_since_previous
32   Workout_033           201.018866
112  Workout_113           117.026655
19   Workout_020            19.979410
116  Workout_117            18.023171
130  Workout_131            12.994583
152  Workout_153             6.010208
101  Workout_102             5.177558
190  Workout_191             4.961551
102  Workout_103             4.818947
193  Workout_194             4.099479


In [42]:
workout_df = workout_df.sort_values(
    "workout_start_time"
).reset_index(drop=True)

In [43]:
workout_df["days_until_next"] = (
    workout_df["workout_start_time"].shift(-1)
    - workout_df["workout_start_time"]
).dt.total_seconds() / (60 * 60 * 24)

In [44]:
workout_df["dropout"] = (
    workout_df["days_until_next"] >= 7
).astype(int)

In [45]:
workout_df.loc[
    workout_df["days_until_next"].isna(),
    "dropout"
] = np.nan

In [46]:
model_df = workout_df.dropna(
    subset=["dropout"]
).copy()

model_df["dropout"] = model_df["dropout"].astype(int)

In [47]:
print(
    model_df["dropout"].value_counts()
)

0    249
1      5
Name: dropout, dtype: int64


In [48]:
print(
    model_df[
        model_df["dropout"] == 1
    ][
        [
            "workout_id",
            "workout_start_time",
            "days_until_next",
            "total_sets",
            "total_exercises",
            "total_reps",
            "total_weight_kg",
            "avg_weight_kg",
            "avg_rpe",
            "total_duration_seconds"
        ]
    ]
)

      workout_id  workout_start_time  days_until_next  total_sets  \
18   Workout_019 2024-01-22 23:47:19        19.979410          14   
31   Workout_032 2024-03-04 23:28:36       201.018866          15   
111  Workout_112 2025-01-21 00:26:44       117.026655          16   
115  Workout_116 2025-05-23 23:44:12        18.023171           4   
129  Workout_130 2025-06-29 00:02:32        12.994583          29   

     total_exercises  total_reps  total_weight_kg  avg_weight_kg  avg_rpe  \
18                 6       260.0            212.5      16.346154      NaN   
31                 6       266.0            161.5      11.535714      NaN   
111                6       315.0            349.0      23.266667      NaN   
115                2        30.0             60.0      20.000000      NaN   
129                9       405.0            540.0      20.000000      NaN   

     total_duration_seconds  
18                    447.0  
31                    322.0  
111                   414.0  
11

In [49]:
features = [
    "total_sets",
    "total_exercises",
    "total_reps",
    "avg_reps",
    "total_weight_kg",
    "avg_weight_kg",
    "avg_rpe",
    "total_distance_meters",
    "total_duration_seconds",
    "muscle_groups",
    "equipment_types",
    "duration_minutes",
    "day_of_week",
    "hour",
    "month",
    "is_weekend"
]

In [ ]:
target = "dropout"

In [ ]:
X = model_df[features]
y = model_df[target]

In [ ]:
print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

In [ ]:
print(
    model_df["dropout"].value_counts()
)

In [ ]:
print(
    model_df[
        model_df["dropout"] == 1
    ][
        [
            "workout_id",
            "workout_start_time",
            "days_until_next"
        ]
    ]
)

In [ ]:
print(
    model_df[
        ["workout_id", "workout_start_time", "days_until_next", "dropout"]
    ].head(25)
)

In [ ]:
print(
    model_df[
        ["workout_id", "workout_start_time", "days_until_next", "dropout"]
    ].tail(10)
)

In [ ]:
print(model_df.shape)
print(model_df["dropout"].value_counts())
print(model_df["dropout"].value_counts(normalize=True))

In [ ]:
features = [
    "total_sets",
    "total_exercises",
    "total_reps",
    "avg_reps",
    "total_weight_kg",
    "avg_weight_kg",
    "avg_rpe",
    "total_distance_meters",
    "total_duration_seconds",
    "muscle_groups",
    "equipment_types",
    "duration_minutes",
    "days_since_previous"
]

X = model_df[features]
y = model_df["dropout"]

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
print(X.isnull().sum())

In [ ]:
X = X.copy()

X["avg_rpe"] = X["avg_rpe"].fillna(X["avg_rpe"].median())
X = X.fillna(0)

print(X.isnull().sum().sum())

# Train the Model

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training:", y_train.value_counts())
print("Testing:", y_test.value_counts())

In [ ]:
from xgboost import XGBClassifier

# Calculate class weight
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

print("Scale pos weight:", scale_pos_weight)

model = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42
)

model.fit(X_train, y_train)

print("Model training completed!")


In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Predictions:")
print(y_pred)

print("\nDropout probabilities:")
print(y_prob)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    zero_division=0
))

In [ ]:
import pandas as pd

importance = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print(importance)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

plt.barh(
    importance["feature"],
    importance["importance"]
)

plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("XGBoost Feature Importance")
plt.gca().invert_yaxis()

plt.show()

In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

roc_auc = roc_auc_score(y_test, y_prob)
pr_auc = average_precision_score(y_test, y_prob)

precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print("ROC-AUC:", round(roc_auc, 4))
print("PR-AUC:", round(pr_auc, 4))
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1 Score:", round(f1, 4))

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training:", y_train.value_counts())
print("Testing:", y_test.value_counts())